# Boundary Element Model

The Boundary Element Model (BEM) delimits the different compartments of the head (brain, skull, skin) by generating 3 different surfaces:
- the inner skull
- the outer skull
- the outer skin

The conductivity between these layers is homogeneous and anisotropic. <br>
The layers are obtained using FreeSurfer's watershed algorithm.

In [1]:
import mindmap as mm 
from pathlib import Path
import mne

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- FREESURFER reconstruction folder.
    - T1

🛠️ **<font color=violet>Configuration</font>**:
- `fconfig`: Configuration file.

🛠️ **<font color=violet>BEM properties</font>**
- `min_skull_thick`: Minimum skull thickness in [mm] to apply.
- `conductivities`: electrical conductivity between head layers.

🛠️ **<font color=violet>Other options</font>**:
- `overwrite`: If True overwrite old files. If False, skip subjects already having files.

In [ ]:
########################## parameters #############################
# Configuration
fconfig = "../configurations/spike.yml"

# BEM
min_skull_thick = 2
conductivities  = [0.3, 0.006, 0.3]

# Run options
overwrite = True
###################################################################

reader   = mm.dataset_reader(fconfig)
mm_dir   = reader.config["MM_DIR"]
subjects = reader.config["SUBS"]


# file names
fbem = lambda sub: mm_dir/sub/"fwd"/f"{sub}_bem.fif" 

## ♾️ Loop

📤 **<font color=pink>Outputs</font>** (files in freesurfer reconstruction bem folder)
- `outer_skin.surf`: outer skin surface (scalp)
- `outer_skull_raw.surf`: outer skull surface
- `inner_skull.surf`: inner skull surface
- `brain.surf`: copy of brain surface
- `inner_skull_raw.surf`: if inner skull going out of outer skull (watershed bug), push outside vertices back inside. This file is the original (bugged) surface
- `inner_skull_changes.vtk`: shows changes between raw inner skull and corrected inner skull

In [4]:
for sub in subjects[28:]: 
    # check file existence 
    fname = fbem(sub)
    if fname.exists() and not overwrite:
        print(f"\033[93mConductor model already exists for subject {sub}\033[0m")
        continue 
    elif not fname.parent.exists():
        fname.mkdir(parents=True)

    # create conductor model
    try:
        bem = mm.solve_volume_conduction(sub, method="bem", min_skull_thick=min_skull_thick, conductivities=conductivities)
    except Exception as e:
        print(f"\033[91mFailed to generate CST for subject {sub} due to: {e}\033[0m")
        continue

    # save 
    mne.write_bem_solution(fname, bem, overwrite=overwrite, verbose="warning")
    print(f"\033[92mSuccessfully saved BEM solution for subject {sub}\033[0m")

Failed to generate CST for subject sub-pat29 due to: Surface inner skull is not completely inside surface outer skull


ERROR:root:No points/cells to operate on
2025-04-04 07:49:56.823 (57724.527s) [    7343FF35E740]vtkIntersectionPolyData:2423  WARN| No Intersection between objects 
ERROR:root:No points/cells to operate on
2025-04-04 07:49:56.823 (57724.527s) [    7343FF35E740]vtkDistancePolyDataFilt:73     ERR| vtkDistancePolyDataFilter (0x5b72effab090): No points/cells to operate on
2025-04-04 07:49:56.824 (57724.528s) [    7343FF35E740]vtkDistancePolyDataFilt:73     ERR| vtkDistancePolyDataFilter (0x5b72effab090): No points/cells to operate on


Successfully saved BEM solution for subject sub-pat30


ERROR:root:No points/cells to operate on
2025-04-04 07:51:19.583 (57807.287s) [    7343FF35E740]vtkIntersectionPolyData:2423  WARN| No Intersection between objects 
2025-04-04 07:51:19.584 (57807.288s) [    7343FF35E740]vtkDistancePolyDataFilt:73     ERR| vtkDistancePolyDataFilter (0x5b72effab090): No points/cells to operate on
ERROR:root:No points/cells to operate on
2025-04-04 07:51:19.584 (57807.288s) [    7343FF35E740]vtkDistancePolyDataFilt:73     ERR| vtkDistancePolyDataFilter (0x5b72effab090): No points/cells to operate on


Successfully saved BEM solution for subject sub-pat31


ERROR:root:No points/cells to operate on
2025-04-04 07:52:37.820 (57885.524s) [    7343FF35E740]vtkIntersectionPolyData:2423  WARN| No Intersection between objects 
ERROR:root:No points/cells to operate on
2025-04-04 07:52:37.820 (57885.524s) [    7343FF35E740]vtkDistancePolyDataFilt:73     ERR| vtkDistancePolyDataFilter (0x5b72ef38ed20): No points/cells to operate on
2025-04-04 07:52:37.821 (57885.525s) [    7343FF35E740]vtkDistancePolyDataFilt:73     ERR| vtkDistancePolyDataFilter (0x5b72ef38ed20): No points/cells to operate on


... (15 outputs left out)

ERROR:root:No points/cells to operate on
2025-04-04 08:09:08.710 (58876.414s) [    7343FF35E740]vtkIntersectionPolyData:2423  WARN| No Intersection between objects 
2025-04-04 08:09:08.711 (58876.415s) [    7343FF35E740]vtkDistancePolyDataFilt:73     ERR| vtkDistancePolyDataFilter (0x5b72f2382700): No points/cells to operate on
ERROR:root:No points/cells to operate on
2025-04-04 08:09:08.711 (58876.416s) [    7343FF35E740]vtkDistancePolyDataFilt:73     ERR| vtkDistancePolyDataFilter (0x5b72f2382700): No points/cells to operate on


Successfully saved BEM solution for subject sub-pat44


ERROR:root:No points/cells to operate on
2025-04-04 08:10:39.077 (58966.781s) [    7343FF35E740]vtkIntersectionPolyData:2423  WARN| No Intersection between objects 
2025-04-04 08:10:39.078 (58966.782s) [    7343FF35E740]vtkDistancePolyDataFilt:73     ERR| vtkDistancePolyDataFilter (0x5b72f2382700): No points/cells to operate on
ERROR:root:No points/cells to operate on
2025-04-04 08:10:39.079 (58966.783s) [    7343FF35E740]vtkDistancePolyDataFilt:73     ERR| vtkDistancePolyDataFilter (0x5b72f2382700): No points/cells to operate on


Successfully saved BEM solution for subject sub-pat45


# Step Wise Run

Run the code step by step for a single subject.

## 📦 Imports

In [1]:
from pathlib import Path 
import mindmap as mm
from mindmap.utils.mri import correct_skull
import mne

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- FREESURFER reconstruction folder.
    - T1

🛠️ **<font color=violet>Configuration</font>**:
- `fconfig`: Configuration file.
- `sub`: Subject id (position in list SUBS)

In [2]:
############# Parameters ###############
fconfig = "../../papers/simf/configuration.yml"
sub     = 1
########################################

config = mm.io.read_config(fconfig)
sub    = config["SUBS"][sub-1]

# directories
fs_dir = config["FS_DIR"]
mm_dir = config["MM_DIR"]

## Boundary Element Model

#### ⚙️ **<font color=green>Create BEM surfaces</font>**

📤 **<font color=pink>Outputs</font>** (files in freesurfer reconstruction bem folder)
- `outer_skin.surf`: outer skin surface (scalp)
- `outer_skull_raw.surf`: outer skull surface
- `inner_skull.surf`: inner skull surface
- `brain.surf`: copy of brain surface
- `inner_skull_raw.surf`: if inner skull going out of outer skull (watershed bug), push outside vertices back inside. This file is the original (bugged) surface
- `inner_skull_changes.vtk`: shows changes between raw inner skull and corrected inner skull

In [3]:
mne.bem.make_watershed_bem(
    subject = sub,
    subjects_dir = fs_dir,
    volume = "T1",
    overwrite = True,
    verbose = "warning",
)

🌟 **Visualize surfaces** ???

#### ⚙️ **<font color=green>Correct surfaces</font>**
The watershed algorithm sometimes generates an inner skull that extends beyond the outer skull. We have implemented some functions to correct this issue.

🛠️ **<font color=violet>Skull properties</font>**:
- `min_skull_thick`: Minimum skull thickness in [cm] to apply during correction.

**Notes:**
Correct skull sends and "ERROR:root:No points/cells to operate on" when there are no correction to be made. The program will continue without stopping.

In [4]:
#################### Parameters ####################
min_skull_thick = 2
####################################################

correct_skull(
    subject     = sub,
    offset      = min_skull_thick,
    fs_dir      = fs_dir,
    save_arrows = True,
)

#### ⚙️ **<font color=green>Create Conduction Model</font>**
In this section, we apply topological checks to verify if the surfaces are compatible with each other in order and apply electrical conductivities to each compartment

📤 **<font color=pink>Outputs</font>**
- Conductor Model (`head_model`)

🛠️ **<font color=violet>Model properties</font>**:
- `conductivities`: The 3 conductivities indicating the electrical conductivity in [S/cm] in each compartment.
    - The first value indicates the conductivity of the brain tissue.
    - The second value indicates the conductivity of the skull tissue.
    - The third value indicates the conductivity of the skin tissue.
- `resolution`: The resolution can be optimized for various subsampling level (`None`, `ico5`=20484, `ico4`=5120, `ico3`=1280). Subsampling using `ico5` or `None` contains to many datapoints and is unable to save. <br>

In [5]:
############# Parameters ################
# properties
conductivities = [0.3, 0.006, 0.3]
resolution     = 4 
#########################################

model = mne.make_bem_model(
        subject      = sub,
        subjects_dir = fs_dir,
        conductivity = conductivities,
        ico          = resolution,
        verbose      = "warning",
)

bem  = mne.make_bem_solution(model, verbose="warning")

💾 **Save BEM** <br>
🛠️ **<font color=violet>Options</font>**:
- `overwrite`: Overwrite head model if already exists.
- `purpose`: Purpose of model (`inv` for solving inverse problem, `sim` for generating simulations). Will adapt the naming of the file.

In [ ]:
######### Parameters ############
# options
overwrite = True 
purpose   = "inv"
#################################

if purpose == "inv":
    fbem = mm_dir/sub/"fwd"/f"{sub}_bem.fif"
elif purpose == "sim":
    fbem = mm_dir/sub/"fwd"/f"{sub}_sim_bem.fif"
else:
    raise ValueError(f"Uknown purpose '{purpose}'")

if not fbem.parent.exists():
    fbem.parent.mkdir(parents=True)

mne.write_bem_solution(fbem, bem, overwrite=overwrite, verbose="warning")
print(f"\033[92mSuccessfully saved {purpose} BEM solution for subject {sub}\033[0m")


Successfully saved inv BEM solution for subject vepcon_sub-10
